In [2]:
import math
bytes_per_weight = 4




In [3]:
def napkin_math(vocab_size, context_length, num_layers, d_model, num_heads, d_ff = None):
    if d_ff == None:
        d_ff = math.ceil(d_model * (8/3) * (1/64)) * 64

    params_embedding = vocab_size * d_model
    params_per_MHA = (d_model**2) * 4 # W_Q, W_K, W_V, W_O
    params_MHA = params_per_MHA * num_layers 
    params_per_FF = d_model * d_ff * 3
    params_FF = params_per_FF * num_layers 
    params_RMS_per_head = d_model * 2
    params_RMS = params_RMS_per_head * num_layers + d_model
    params_final_lin = d_model * vocab_size
    params_total = params_RMS + params_MHA + params_FF + params_final_lin + params_embedding

    muls_per_MHA = 4 * d_model**2 * context_length + 2 * d_model * context_length**2
    muls_per_FF = d_ff * d_model * context_length * 3
    muls_final_layer = context_length * d_model * vocab_size
    total_muls = (muls_per_FF+muls_per_MHA) * num_layers + muls_final_layer
    total_flops = total_muls * 2

    gflop_total = total_flops / 1e9
    gflop_MHA_attention = 2 * 2 * d_model * context_length**2 * num_layers / 1e9
    gflop_MHA_other = 2 * 4 * d_model**2 * context_length * num_layers / 1e9
    gflop_FF = d_ff * d_model * context_length * 3 * num_layers * 2 / 1e9
    gflop_final_lin = muls_final_layer * 2 / 1e9


    
    print(f"vocab_size: {vocab_size}\n"
      f"context_length: {context_length}\n"
      f"num_layers: {num_layers}\n"
      f"d_model: {d_model}\n"
      f"num_heads: {num_heads}\n"
      f"d_ff: {d_ff}\n\n"
      f"total params: {params_total:.3e}\n"
      f"    params embedding: {params_embedding:.3e}, {params_embedding/params_total:.2%}\n"
      f"    params MHA: {params_MHA:.3e}, {params_MHA/params_total:.2%}\n"
      f"    params FF: {params_FF:.3e}, {params_FF/params_total:.2%}\n"
      f"    params RMS: {params_RMS:.3e}, {params_RMS/params_total:.2%}\n"
      f"    params final lin: {params_final_lin:.3e}, {params_final_lin/params_total:.2%}\n"
      f"total gflop/forward pass: {gflop_total : .2f}\n"
      f"    MHA attention (d_model * context_len^2 * stuff): {gflop_MHA_attention : .2f}, {gflop_MHA_attention/gflop_total:.2%} \n"
      f"    MHA other (d_model^2 * context_len * stuff): {gflop_MHA_other:.2f}, {gflop_MHA_other/gflop_total:.2%} \n"
      f"    FF: {gflop_FF:.2f}, {gflop_FF/gflop_total:.2%}\n"
      f"    Final layer: {gflop_final_lin:.2f}, {gflop_final_lin/gflop_total:.2%}\n")

In [4]:
napkin_math(vocab_size = 50257, 
            context_length = 1024, 
            num_layers = 48, 
            d_model = 1600, 
            num_heads = 25)

print("\nGPT2 Small:")
napkin_math(vocab_size = 50257, 
            context_length = 1024, 
            num_layers = 12, 
            d_model = 768, 
            num_heads = 12)

print("\nGPT2 Medium:")
napkin_math(vocab_size = 50257, 
            context_length = 1024, 
            num_layers = 24, 
            d_model = 1024, 
            num_heads = 16)

print("\nGPT2 Large:")
napkin_math(vocab_size = 50257, 
            context_length = 1024, 
            num_layers = 36, 
            d_model = 1280, 
            num_heads = 20)


print("\nGPT2 XL")
napkin_math(vocab_size = 50257, 
            context_length = 1024, 
            num_layers = 48, 
            d_model = 1600, 
            num_heads = 25)

print("\nGPT2 XL big context")
napkin_math(vocab_size = 50257, 
            context_length = 16384, 
            num_layers = 48, 
            d_model = 1600, 
            num_heads = 25)



vocab_size: 50257
context_length: 1024
num_layers: 48
d_model: 1600
num_heads: 25
d_ff: 4288

total params: 1.640e+09
    params embedding: 8.041e+07, 4.90%
    params MHA: 4.915e+08, 29.96%
    params FF: 9.880e+08, 60.22%
    params RMS: 1.552e+05, 0.01%
    params final lin: 8.041e+07, 4.90%
total gflop/forward pass:  3516.77
    MHA attention (d_model * context_len^2 * stuff):  322.12, 9.16% 
    MHA other (d_model^2 * context_len * stuff): 1006.63, 28.62% 
    FF: 2023.33, 57.53%
    Final layer: 164.68, 4.68%


GPT2 Small:
vocab_size: 50257
context_length: 1024
num_layers: 12
d_model: 768
num_heads: 12
d_ff: 2048

total params: 1.621e+08
    params embedding: 3.860e+07, 23.80%
    params MHA: 2.831e+07, 17.46%
    params FF: 5.662e+07, 34.92%
    params RMS: 1.920e+04, 0.01%
    params final lin: 3.860e+07, 23.80%
total gflop/forward pass:  291.65
    MHA attention (d_model * context_len^2 * stuff):  38.65, 13.25% 
    MHA other (d_model^2 * context_len * stuff): 57.98, 19.88% 
 